# NutriPulse - Partie 1 : Nettoyage des données MICS6 Togo 2017

**Ce que fait ce notebook :** il lit les trois fichiers bruts de l'enquête (enfants, ménages, membres des ménages), les assemble, retire les mesures douteuses, crée les deux variables à prédire (retard de croissance et émaciation), remet chaque variable au propre, puis écrit **un seul fichier propre** : `data/clean/nutripulse_clean.parquet`.

**Ce qu'il ne fait pas, volontairement :** pas d'analyse, pas de graphiques, pas de modèle. Il ne remplace pas non plus les valeurs manquantes, n'encode pas les catégories et ne change pas les échelles : tout cela "apprend" sur les données et sera fait plus tard, au moment de la modélisation, pour éviter que de l'information du test ne fuite dans l'entraînement.

**Comment lire ce notebook :** chaque étape suit le même schéma, *Quoi* (ce qu'on fait), *Pourquoi* (la raison), puis le code et son résultat. Rien n'est caché : si une règle change, on la corrige ici, on relance tout, et le fichier propre est régénéré. On ne retouche jamais le fichier propre à la main.

**Règles de base**
- Les fichiers de `data/raw` ne sont jamais modifiés.
- Seules les variables vraiment utiles sont gardées.
- Quand un code spécial apparaît (ne sait pas, incohérent...), on lit d'abord son sens dans l'enquête avant de décider.
- Les contrôles automatiques de la fin font échouer le notebook si quelque chose cloche.

**Sommaire**
1. Lire les fichiers et comprendre les codes
2. Vérifier les identifiants
3. Sélectionner les colonnes utiles
4. Assembler les trois fichiers
5. Retirer les mesures absentes ou douteuses
6. Créer les deux cibles
7. Nettoyer les variables une par une
8. Mettre en place les groupes, les poids et les colonnes d'audit
9. Fixer les types
10. Contrôles automatiques
11. Exporter et relire le fichier propre

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

# Le notebook peut être lancé depuis le dossier notebook/ ou depuis la racine du projet
RACINE = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DOSSIER_BRUT = RACINE / "data" / "raw"
DOSSIER_PROPRE = RACINE / "data" / "clean"
DOSSIER_PROPRE.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
print("Racine du projet :", RACINE)

Racine du projet : C:\Users\HP ELITEBOOK\OneDrive\Documents\NutriPulse_push


## Étape 1 - Lire les fichiers en gardant les codes

**Quoi.** On lit `ch`, `hh` et `hl` en gardant les **codes numériques** (1, 2, 3...) et non les textes ("Oui", "Non"...). On garde aussi les métadonnées, c'est-à-dire la liste des libellés de chaque code.

**Pourquoi.** On veut voir les codes spéciaux (7, 8, 9, 96, 97...) pour décider nous-mêmes, variable par variable, s'il s'agit d'une réponse manquante ou d'une vraie réponse.

- `ch` : un enfant de moins de 5 ans par ligne (mesures, z-scores, instruction de la mère...)
- `hh` : un ménage par ligne (électricité, toilettes, eau, animaux...)
- `hl` : une personne par ligne (sexe, âge de chaque membre du ménage)

In [2]:
def lire(nom):
    df, meta = pyreadstat.read_sav(DOSSIER_BRUT / f"{nom}.sav", apply_value_formats=False)
    return df, meta

ch, meta_ch = lire("ch")
hh, meta_hh = lire("hh")
hl, meta_hl = lire("hl")

print(f"ch : {ch.shape[0]:>6} lignes, {ch.shape[1]:>3} colonnes  (enfants)")
print(f"hh : {hh.shape[0]:>6} lignes, {hh.shape[1]:>3} colonnes  (ménages)")
print(f"hl : {hl.shape[0]:>6} lignes, {hl.shape[1]:>3} colonnes  (personnes)")

ch :   5030 lignes, 467 colonnes  (enfants)
hh :   8404 lignes, 297 colonnes  (ménages)
hl :  34988 lignes,  80 colonnes  (personnes)


### Lire le sens des codes avant toute décision

**Pourquoi.** Un même code n'a pas toujours le même sens. Par exemple, le code 97 veut dire "incohérent" pour la taille du ménage, mais "pas de repas préparé" (une vraie réponse) pour le type de cuisinière. On ne devine jamais : on lit.

La petite fonction ci-dessous affiche le sens des codes d'une variable.

In [3]:
def libelles(meta, variable):
    print(f"{variable} - {meta.column_names_to_labels[variable]}")
    for code_, texte in meta.variable_value_labels.get(variable, {}).items():
        print(f"     {int(code_):>3} = {texte}")
    print()

print("===== Fichier ch (enfants) =====\n")
for v in ["HH7", "melevel", "BR1", "HH6", "windex5", "ethnicity", "HAZFLAG", "WHZFLAG","HL4"]:
    libelles(meta_ch, v)

===== Fichier ch (enfants) =====

HH7 - Région
       1 = MARITIME
       2 = PLATEAUX
       3 = CENTRALE
       4 = KARA
       5 = SAVANES
       6 = LOME COMMUNE
       7 = GOLFE URBAIN

melevel - Niveau d'instruction de la mère
       0 = Aucun/Préscolaire
       1 = Primaire
       2 = Secondaire et plus
       9 = NSP/Manquant

BR1 - Certificat de naissance
       1 = OUI, VU
       2 = OUI, PAS VU
       3 = NON
       7 = INCOHERENT
       8 = NSP
       9 = NON REPONSE

HH6 - Milieu
       1 = URBAIN
       2 = RURAL

windex5 - Quintile du bien être
       1 = Le plus pauvre
       2 = Second
       3 = Moyen
       4 = Quatrième
       5 = Le plus riche

ethnicity - Ethnie du chef de ménage
       1 = Adja-Ewe
       2 = Kabye-Tem
       3 = Paragourma
       4 = Ana-Ife
       5 = Akposso/Akébou
       6 = Autres togolais
       7 = Autres nationalités
       9 = NSP/Manquant

HAZFLAG - Height for age flag WHO
       0 = No error
       1 = Error flag

WHZFLAG - Weight for 

In [4]:
print("===== Fichier hh (ménages) =====\n")
for v in ["HC8", "EU1", "WS1", "WS11", "HW1", "HC12", "HC17", "HH48"]: # Ajouter HC2 pour voir
    libelles(meta_hh, v)

===== Fichier hh (ménages) =====

HC8 - Ménage a l'électricité
       1 = Oui, connecté au système général
       2 = Oui, en dehors du système (Générateur/Système isolé)
       3 = Non
       4 = Incohérent

EU1 - Type de cuisinière principale
       1 = Cuisinière électrique
       2 = Cuisinière solaire
       3 = Cuisinièree à gaz de pétrole liquefié (GPL)
       4 = Cuisinière à gaz naturel
       5 = Cuisinière à Boigaz
       6 = Cuisinière à combustible liquide
       7 = Cuisinière à combustible solide
       8 = Cuisinière locale à combustible solide
       9 = Feu sur trois pièrres / Feu ouvert
      96 = Autre
      97 = Pas de repas preparé dans le ménage

WS1 - Source principale d'eau de boisson
      11 = Robinet: dans le logement
      12 = Robinet: dans la concession / Jardin / Parcelle
      13 = Robinet: chez le voisin
      14 = Robinet: robinet publics / Borne fontaine / Kiosque à eau
      21 = Puits à pompe / Forage
      31 = Puits creusé: protègé
      32 = Pui

**Deux pièges repérés à la lecture :**

1. Le code 97 n'a pas le même sens partout (voir plus haut : "incohérent" pour la taille du ménage, vraie réponse pour la cuisinière).
2. **Les libellés de l'ethnie ne sont pas les mêmes dans `ch` et dans `hh`.** Regardons les deux :

In [5]:
libelles(meta_ch, "ethnicity")
libelles(meta_hh, "HC2")

ethnicity - Ethnie du chef de ménage
       1 = Adja-Ewe
       2 = Kabye-Tem
       3 = Paragourma
       4 = Ana-Ife
       5 = Akposso/Akébou
       6 = Autres togolais
       7 = Autres nationalités
       9 = NSP/Manquant

HC2 - Groupe ethnique du chef de ménage
       1 = ADJA /EWE
       2 = KABYE /TEM
       3 = AKPOSSO / AKEBOU
       4 = IFE/ ANA
       5 = PARA / GOURMA /AKAN
      96 = AUTRES TOGOLAIS
      97 = AUTRES NATIONALITES



Pour les codes 1, 2 et 4, les deux fichiers disent la même chose. Mais le code 3 est "Paragourma" dans `ch` et "Akposso" dans `hh`, et c'est l'inverse pour le code 5. (Les codes 6 et 7 de `ch` correspondent aux codes 96 et 97 de `hh` : "autres Togolais" et "autres nationalités".)

Il faut donc trancher : **quels libellés sont les bons ?** On le vérifie avec la géographie. Les Para-Gourma vivent surtout dans les Savanes (nord), les Akposso surtout dans les Plateaux. Regardons où se trouvent les enfants des codes 3 et 5 :

In [6]:
region_nom = ch["HH7"].map(meta_ch.variable_value_labels["HH7"])
for code_ in [3, 5]:
    part = region_nom[ch["ethnicity"] == code_].value_counts(normalize=True).head(2).round(2)
    print(f"Code {code_} - régions les plus fréquentes :", part.to_dict())

Code 3 - régions les plus fréquentes : {'PLATEAUX': 0.61, 'GOLFE URBAIN': 0.17}
Code 5 - régions les plus fréquentes : {'SAVANES': 0.7, 'KARA': 0.18}


Le code 5 se trouve surtout dans les Savanes et le code 3 surtout dans les Plateaux. **Les libellés de `hh` sont donc les bons** (3 = Akposso-Akébou, 5 = Para-Gourma-Akan) et ceux de `ch` sont inversés pour ces deux codes. On utilisera la colonne d'ethnie de `ch` (les codes sont les mêmes), mais avec les libellés de `hh`. L'ethnie ne servant qu'à l'audit d'équité, l'enjeu reste limité, mais autant qu'elle soit juste.

## Étape 2 - Vérifier les identifiants

**Quoi.** On vérifie qu'aucun identifiant n'apparaît en double :
- un **enfant** se reconnaît par (grappe, ménage, numéro de ligne de l'enfant) ;
- un **ménage** par (grappe, ménage) ;
- une **personne** par (grappe, ménage, numéro de ligne).

**Pourquoi.** Un doublon d'identifiant ferait multiplier des lignes au moment de l'assemblage, sans qu'on s'en rende compte. Une "grappe" est un petit groupe de ménages enquêtés dans la même zone.

In [7]:
assert not ch.duplicated(["HH1", "HH2", "LN"]).any(), "Doublon d'enfant dans ch"
assert not hh.duplicated(["HH1", "HH2"]).any(), "Doublon de ménage dans hh"
assert not hl.duplicated(["HH1", "HH2", "HL1"]).any(), "Doublon de personne dans hl"

# Dans ch, LN et UF3 doivent désigner la même chose : le numéro de ligne de l'enfant dans le ménage
assert (ch["LN"] == ch["UF3"]).all()
print("Aucun doublon. Les identifiants sont bien uniques.")

Aucun doublon. Les identifiants sont bien uniques.


## Étape 3 - Sélectionner les colonnes utiles et les renommer

**Quoi.** Les fichiers ont des centaines de colonnes (467 pour `ch`). On garde seulement celles dont on a besoin, avec des noms courts, lisibles et en minuscules.

**Pourquoi.** Plusieurs colonnes portent le même nom d'un fichier à l'autre (par exemple `HH7`, la région, existe dans `ch` et dans `hh`). Choisir et renommer **avant** l'assemblage évite les doublons de colonnes et les confusions.

Les colonnes gardées sont de cinq sortes : identifiants, mesures (pour construire les cibles), variables d'entrée (celles qui serviront au modèle), poids d'enquête, et colonnes d'audit (pour vérifier plus tard l'équité).

In [8]:
# ---- Fichier des enfants
colonnes_ch = {
    "HH1": "grappe",
    "HH2": "menage",
    "LN": "ligne_enfant",
    "UF4": "ligne_gardien",          # numéro de ligne (dans hl) de la personne qui garde l'enfant
    "CAGE": "age_mois",
    "HAZ2": "haz",                   # taille pour l'âge, normes OMS
    "WHZ2": "whz",                   # poids pour la taille, normes OMS
    "HAZFLAG": "haz_flag",           # 1 = mesure jugée implausible
    "WHZFLAG": "whz_flag",           # 1 = mesure jugée implausible
    "melevel": "edu_mere",
    "BR1": "acte_naissance",
    "HH7": "region_ch",              # copie, servira seulement de contrôle
    "HL4": "sexe_ch",                # copie, servira seulement de contrôle
    "HH6": "milieu",
    "windex5": "quintile_richesse",
    "ethnicity": "ethnie",
    "chweight": "poids_enfant",
}
enfants = ch[list(colonnes_ch)].rename(columns=colonnes_ch)

# ---- Fichier des ménages
colonnes_hh = {
    "HH1": "grappe",
    "HH2": "menage",
    "HH7": "region",
    "HC8": "electricite",
    "EU1": "cuisiniere",
    "WS1": "source_eau",
    "WS11": "toilettes",
    "HW1": "lavage_mains",
    "HC12": "telephone",
    "HC17": "animaux",
    "HH48": "taille_menage",
}
menages = hh[list(colonnes_hh)].rename(columns=colonnes_hh)

# ---- Fichier des personnes
colonnes_hl = {"HH1": "grappe", "HH2": "menage", "HL1": "ligne", "HL4": "sexe", "HL6": "age"}
personnes = hl[list(colonnes_hl)].rename(columns=colonnes_hl)

print(f"Enfants   : {enfants.shape[1]} colonnes gardées sur {ch.shape[1]}")
print(f"Ménages   : {menages.shape[1]} colonnes gardées sur {hh.shape[1]}")
print(f"Personnes : {personnes.shape[1]} colonnes gardées sur {hl.shape[1]}")

Enfants   : 17 colonnes gardées sur 467
Ménages   : 11 colonnes gardées sur 297
Personnes : 5 colonnes gardées sur 80


## Étape 4 - Assembler les trois fichiers

**Quoi.** On part de la table des enfants et on lui ajoute trois informations :
1. le **sexe de l'enfant**, cherché dans `hl` grâce au numéro de ligne de l'enfant ;
2. les **caractéristiques du ménage** (région, électricité, toilettes...), cherchées dans `hh` grâce à (grappe, ménage) ;
3. l'**âge de la personne qui garde l'enfant**, cherché dans `hl` avec le numéro de ligne du gardien.

**Pourquoi.** L'information utile pour un enfant est répartie à trois endroits. Toutes les jointures conservent tous les enfants : on préfère voir apparaître une case vide plutôt que perdre un enfant sans le savoir. De plus, l'option `validate` de pandas arrête le notebook si une jointure créait des lignes en double.

**Contrôles.** Le nombre de lignes doit être identique avant et après, et les copies de contrôle (sexe et région sont déjà présents dans `ch`) doivent correspondre à 100 % à ce que la jointure a apporté.

In [9]:
nb_avant = len(enfants)

# 1) Sexe de l'enfant : on le cherche dans hl, à la ligne de l'enfant
sexes = personnes[["grappe", "menage", "ligne", "sexe"]].rename(
    columns={"ligne": "ligne_enfant", "sexe": "sexe_hl"})
enfants = enfants.merge(sexes, on=["grappe", "menage", "ligne_enfant"],
                        how="left", validate="one_to_one")

# 2) Caractéristiques du ménage
enfants = enfants.merge(menages, on=["grappe", "menage"],
                        how="left", validate="many_to_one")

# 3) Âge de la personne qui garde l'enfant : une autre ligne de hl
gardiens = personnes[["grappe", "menage", "ligne", "age"]].rename(
    columns={"ligne": "ligne_gardien", "age": "age_gardien"})
enfants = enfants.merge(gardiens, on=["grappe", "menage", "ligne_gardien"],
                        how="left", validate="many_to_one")

# ---- Contrôles
assert len(enfants) == nb_avant, "Des lignes ont été ajoutées ou perdues pendant l'assemblage"
assert not enfants.duplicated(["grappe", "menage", "ligne_enfant"]).any()
assert (enfants["sexe_hl"] == enfants["sexe_ch"]).all(), "Le sexe de la jointure diffère de celui de ch"
assert (enfants["region"] == enfants["region_ch"]).all(), "La région de la jointure diffère de celle de ch"

print(f"Lignes avant : {nb_avant} - après : {len(enfants)}")
print("Sexe : 100 % de concordance avec la copie de ch")
print("Région : 100 % de concordance avec la copie de ch")
print("Enfants sans information ménage :", int(enfants["electricite"].isna().sum()))
print("Enfants sans âge de gardien     :", int(enfants["age_gardien"].isna().sum()))

Lignes avant : 5030 - après : 5030
Sexe : 100 % de concordance avec la copie de ch
Région : 100 % de concordance avec la copie de ch
Enfants sans information ménage : 0
Enfants sans âge de gardien     : 0


L'assemblage est correct : aucun enfant perdu ni dupliqué, et les copies de contrôle concordent. Les copies `sexe_ch` et `region_ch` ont rempli leur rôle, on les retire.

In [10]:
enfants = enfants.drop(columns=["sexe_ch", "region_ch"])

## Étape 5 - Retirer les mesures absentes ou douteuses

**Quoi.** On retire deux sortes d'enfants :
- ceux qui n'ont **pas de mesure** (enfant absent, refus, etc.) ;
- ceux dont la mesure est jugée **implausible** par l'enquête (drapeaux de plausibilité, par exemple une taille impossible pour l'âge).

**Pourquoi.** Les cibles sont calculées à partir des mesures. Sans mesure fiable, il n'y a pas de cible. Les mesures implausibles sont de plus codées avec une valeur factice (99.99) qui fausserait tout calcul.

On retire un enfant dès que l'une de ses deux mesures (taille pour l'âge ou poids pour la taille) pose problème, afin de garder **un seul jeu d'enfants** pour les deux modèles.

In [11]:
sans_mesure = enfants["haz"].isna() | enfants["whz"].isna()
douteux = (enfants["haz_flag"] == 1) | (enfants["whz_flag"] == 1)

print(f"Enfants au départ                 : {len(enfants)}")
print(f"Sans mesure                       : {int(sans_mesure.sum())}")
print(f"Avec mesure jugée implausible     : {int((douteux & ~sans_mesure).sum())}")

# La valeur factice 99.99 ne doit se trouver que chez les enfants qu'on va retirer
factice = (enfants["haz"] == 99.99) | (enfants["whz"] == 99.99)
assert (factice <= (douteux | sans_mesure)).all()

enfants = enfants[~(sans_mesure | douteux)].reset_index(drop=True)
nb_enfants = len(enfants)
print(f"Enfants gardés                    : {nb_enfants}")

assert enfants[["haz", "whz"]].notna().all().all()
assert enfants[["haz", "whz"]].abs().max().max() < 10, "Il reste une valeur factice"
print("Plus aucune valeur factice (99.99) dans les mesures.")

Enfants au départ                 : 5030
Sans mesure                       : 88
Avec mesure jugée implausible     : 42
Enfants gardés                    : 4900
Plus aucune valeur factice (99.99) dans les mesures.


## Étape 6 - Créer les deux cibles

**Quoi.** On utilise les z-scores de l'OMS (jamais les anciens z-scores NCHS) et on les compare aux seuils de l'OMS :

- **z-score supérieur ou égal à -2** : normal
- **z-score entre -3 et -2** : modéré
- **z-score inférieur à -3** : sévère

On crée :
- `stunt` : 1 si le retard de croissance (taille pour l'âge) est sous -2, sinon 0 ;
- `waste` : 1 si l'émaciation (poids pour la taille) est sous -2, sinon 0 ;
- `stunt_niveau` et `waste_niveau` : 0 normal, 1 modéré, 2 sévère. Ils servent à **décrire** la réalité, pas à entraîner un modèle.

On garde aussi `haz` et `whz` (les z-scores) pour les graphiques et les vérifications.

**Pourquoi.** `stunt` et `waste` seront les deux réponses que les modèles apprendront à prédire. Les z-scores, eux, ne doivent **jamais** servir d'entrée : ils sont la réponse elle-même.

Ce qui suit est une vérification de la construction, pas une analyse.

In [12]:
def niveau(z):
    # 2 = sévère (sous -3), 1 = modéré (entre -3 et -2), 0 = normal (-2 ou plus)
    return np.select([z < -3, z < -2], [2, 1], default=0).astype("int8")

enfants["stunt_niveau"] = niveau(enfants["haz"])
enfants["waste_niveau"] = niveau(enfants["whz"])
enfants["stunt"] = (enfants["haz"] < -2).astype("int8")
enfants["waste"] = (enfants["whz"] < -2).astype("int8")

print("Retard de croissance (0 normal, 1 modéré, 2 sévère) :")
print(enfants["stunt_niveau"].value_counts().sort_index().to_string())
print()
print("Émaciation (0 normal, 1 modérée, 2 sévère) :")
print(enfants["waste_niveau"].value_counts().sort_index().to_string())


print("Retard de croissance (0 non, 1 oui) :")
print(enfants["stunt"].value_counts().sort_index().to_string())
print()
print("Émaciation (0 non, 1 oui) :")
print(enfants["waste"].value_counts().sort_index().to_string())

Retard de croissance (0 normal, 1 modéré, 2 sévère) :
stunt_niveau
0    3708
1     838
2     354

Émaciation (0 normal, 1 modérée, 2 sévère) :
waste_niveau
0    4618
1     225
2      57
Retard de croissance (0 non, 1 oui) :
stunt
0    3708
1    1192

Émaciation (0 non, 1 oui) :
waste
0    4618
1     282


## Étape 7 - Nettoyer les variables une par une

Les modèles utiliseront **14 variables d'entrée** : 6 communes aux deux modèles, 4 propres au retard de croissance, 4 propres à l'émaciation.

**Règles générales appliquées partout :**
- les codes "incohérent", "ne sait pas", "non réponse" deviennent des **cases vides** ;
- les codes trop détaillés sont **regroupés** (cuisinière, toilettes, lavage des mains, source d'eau) pour éviter de petites catégories qui n'apporteraient que du bruit ;
- **`les questions oui/non sont recodées en 1 / 0`** ;
- un code spécial laissé tel quel serait lu comme une vraie quantité par un modèle, d'où l'importance de les transformer.

On garde une copie des colonnes brutes (`brut`) pour pouvoir, à la fin, vérifier que le nombre de cases vides correspond exactement à ce qui était attendu.

In [13]:
# Copie des colonnes brutes de l'échantillon, utilisée pour les contrôles de la fin
brut = enfants.copy()

def recoder(serie, correspondance):
    """Transforme les codes selon le dictionnaire. None = case vide.
    Si un code n'est pas prévu dans le dictionnaire, on s'arrête : on ne devine jamais."""
    inconnus = set(serie.dropna().unique()) - set(correspondance)
    assert not inconnus, f"Codes non prévus : {inconnus}"
    return serie.map(correspondance)

def compter(serie):
    print(serie.value_counts(dropna=False).to_string())

**NB** : `Pour les libelles revenir à l'étape 1`.

### Variables communes aux deux modèles

**`age_mois` - âge de l'enfant en mois (0 à 59).** Aucun code spécial dans cette variable : on vérifie seulement que les valeurs sont bien comprises entre 0 et 59.

In [14]:
print("Minimum :", enfants["age_mois"].min(), "- Maximum :", enfants["age_mois"].max(),
      "- Valeurs vides :", int(enfants["age_mois"].isna().sum()))
assert enfants["age_mois"].between(0, 59).all()

Minimum : 0.0 - Maximum : 59.0 - Valeurs vides : 0


**`region` - région de résidence (7 régions).** On remplace les numéros par les noms. C'est une variable **nominale** : les régions n'ont pas d'ordre naturel.

In [15]:
noms_regions = {1: "Maritime", 2: "Plateaux", 3: "Centrale", 4: "Kara",
                5: "Savanes", 6: "Lomé Commune", 7: "Golfe Urbain"}
enfants["region"] = recoder(brut["region"], noms_regions)
compter(enfants["region"])

region
Savanes         1137
Kara             794
Centrale         742
Plateaux         613
Golfe Urbain     589
Maritime         538
Lomé Commune     487


**`edu_mere` - instruction de la mère.** Trois niveaux ordonnés : aucune, primaire, secondaire ou plus. Le code 9 ("ne sait pas") devient une case vide.

In [16]:
enfants["edu_mere"] = recoder(brut["edu_mere"], {
    0: "Aucune", 1: "Primaire", 2: "Secondaire ou plus", 9: None})
compter(enfants["edu_mere"])

edu_mere
Aucune                1893
Primaire              1649
Secondaire ou plus    1357
None                     1


**`electricite` - électricité dans le ménage.** Trois niveaux ordonnés : pas d'électricité, électricité hors réseau (générateur, solaire), électricité du réseau général. Le code 4 ("incohérent") devient une case vide.

In [17]:
enfants["electricite"] = recoder(brut["electricite"], {
    3: "Non", 2: "Hors réseau", 1: "Réseau", 4: None})
compter(enfants["electricite"])

electricite
Non            2548
Réseau         2129
Hors réseau     223


**`cuisiniere` - type de cuisinière.** On regroupe les 11 codes d'origine en **4 modalités** :
- *Électricité, gaz ou liquide* : cuisinière électrique, solaire, à gaz (GPL ou naturel), biogaz, combustible liquide ;
- *Combustible solide (cuisinière)* : cuisinière à combustible solide, y compris les foyers améliorés locaux ;
- *Feu ouvert* : feu sur trois pierres ou feu ouvert ;
- *Pas de repas préparé* : **le code 97 est ici une vraie réponse**, on le garde.

Le code 96 ("autre") est trop flou pour être rangé : il devient une case vide.

In [18]:
enfants["cuisiniere"] = recoder(brut["cuisiniere"], {
    1: "Électricité, gaz ou liquide", 2: "Électricité, gaz ou liquide",
    3: "Électricité, gaz ou liquide", 4: "Électricité, gaz ou liquide",
    5: "Électricité, gaz ou liquide", 6: "Électricité, gaz ou liquide",
    7: "Combustible solide (cuisinière)", 8: "Combustible solide (cuisinière)",
    9: "Feu ouvert",
    97: "Pas de repas préparé",
    96: None})
compter(enfants["cuisiniere"])

cuisiniere
Feu ouvert                         2739
Combustible solide (cuisinière)    1840
Électricité, gaz ou liquide         312
Pas de repas préparé                  7
None                                  2


**À noter :** la modalité *Pas de repas préparé* ne concerne que très peu d'enfants (7). On la garde telle quelle, car c'est une vraie réponse et non une erreur. Si elle reste trop rare au moment de la modélisation, c'est à ce moment-là qu'on décidera de la regrouper avec une autre : le nettoyage ne tranche pas ce qui dépend de l'apprentissage.

**`toilettes` - type de toilettes.** On regroupe les 14 codes d'origine en **4 modalités ordonnées**, de la moins à la plus favorable pour l'hygiène :
1. *Aucune* : pas de toilettes, nature ou champ ;
2. *Non améliorée* : latrine sans dalle, seau, toilettes suspendues, chasse d'eau qui se vide à l'air libre ou vers un lieu inconnu ;
3. *Latrine améliorée* : latrine ventilée ou avec dalle, toilette à compostage, chasse d'eau vers latrine ;
4. *Chasse d'eau* : reliée à un égout ou à une fosse septique.

Les codes 96 ("autre") et 97 ("incohérent") deviennent des cases vides.

In [19]:
enfants["toilettes"] = recoder(brut["toilettes"], {
    95: "Aucune",
    14: "Non améliorée", 18: "Non améliorée", 23: "Non améliorée",
    41: "Non améliorée", 51: "Non améliorée",
    13: "Latrine améliorée", 21: "Latrine améliorée", 22: "Latrine améliorée",
    31: "Latrine améliorée",
    11: "Chasse d'eau", 12: "Chasse d'eau",
    96: None, 97: None})
compter(enfants["toilettes"])

toilettes
Aucune               2559
Latrine améliorée    1071
Chasse d'eau          781
Non améliorée         476
None                   13


### Variables propres au retard de croissance

**`sexe_f` - sexe de l'enfant.** Une variable à deux valeurs : **1 = fille, 0 = garçon**. Le nom `sexe_f rappelle ce que veut dire le 1`.

In [20]:
enfants["sexe_f"] = recoder(brut["sexe_hl"], {2: 1, 1: 0})
compter(enfants["sexe_f"])

sexe_f
0    2496
1    2404


**`acte_naissance` - l'enfant a un acte de naissance.** Les réponses "oui, vu" et "oui, pas vu" sont regroupées en **1** ; "non" devient **0** ; "ne sait pas" devient une case vide.

In [21]:
enfants["acte_naissance"] = recoder(brut["acte_naissance"], {
    1: 1, 2: 1, 3: 0, 7: None, 8: None, 9: None})
compter(enfants["acte_naissance"])

acte_naissance
1.0    3795
0.0    1093
NaN      12


**`eau_amelioree` - source d'eau de boisson.** On passe des 16 sources d'origine à une seule question : la source est-elle **améliorée** (1) ou non (0) ?

- *Améliorée* : robinet (logement, concession, voisin, public), puits à pompe ou forage, puits creusé protégé, source protégée, eau de pluie, camion-citerne, charrette avec citerne, eau en bouteille ou en sachet.
- *Non améliorée* : puits creusé non protégé, source non protégée, eau de surface (rivière, mare, lac...).

Les codes 96 ("autre") et 97 ("incohérent") deviennent des cases vides.

In [22]:
ameliorees = [11, 12, 13, 14, 21, 31, 41, 51, 61, 71, 91, 92]
non_ameliorees = [32, 42, 81]
correspondance_eau = {c: 1 for c in ameliorees}
correspondance_eau.update({c: 0 for c in non_ameliorees})
correspondance_eau.update({96: None, 97: None})
enfants["eau_amelioree"] = recoder(brut["source_eau"], correspondance_eau)
compter(enfants["eau_amelioree"])

eau_amelioree
1.0    3534
0.0    1357
NaN       9


**`animaux` - le ménage possède des animaux d'élevage.** Oui = 1, non = 0, "incohérent" = case vide.

In [23]:
enfants["animaux"] = recoder(brut["animaux"], {1: 1, 2: 0, 7: None})
compter(enfants["animaux"])

animaux
1.0    2878
0.0    2022


### Variables propres à l'émaciation

**`age_gardien` - âge de la personne qui garde l'enfant.** Le code 97 ("incohérent") deviendrait un âge de 97 ans : on le remplace par une case vide. Après l'assemblage, on regarde aussi les valeurs extrêmes pour voir si elles sont plausibles.

In [24]:
enfants["age_gardien"] = brut["age_gardien"].where(brut["age_gardien"] != 97)
print(enfants["age_gardien"].describe().round(1).to_string())

count    4900.0
mean       31.3
std         8.6
min        14.0
25%        25.0
50%        30.0
75%        36.0
max        75.0


Les âges vont d'environ 14 à 75 ans, ce qui est plausible pour une personne qui garde un enfant. **Aucune correction n'est nécessaire** : on ne modifie pas des valeurs qui sont réalistes.

In [25]:
assert enfants["age_gardien"].dropna().between(10, 90).all()

**`taille_menage` - nombre de personnes du ménage.** Le code 97 ("incohérent") devient une case vide. Les très grands ménages sont rares et pourraient peser trop lourd : on **plafonne** la taille à la valeur du 99e percentile (c'est-à-dire la taille que 99 % des enfants ne dépassent pas). Cette valeur est affichée ci-dessous : l'interface finale devra appliquer le même plafond.

In [26]:
taille = brut["taille_menage"].where(brut["taille_menage"] != 97)
plafond_taille_menage = int(taille.quantile(0.99))
enfants["taille_menage"] = taille.clip(upper=plafond_taille_menage)

print("Plafond (99e percentile)        :", plafond_taille_menage, "personnes")
print("Enfants concernés par le plafond:", int((taille > plafond_taille_menage).sum()))
print("Taille maximale avant plafond   :", int(taille.max()))

Plafond (99e percentile)        : 19 personnes
Enfants concernés par le plafond: 19
Taille maximale avant plafond   : 23


**`lavage_mains` - lieu de lavage des mains observé.** On regroupe en **3 modalités ordonnées** :
1. *Aucun lieu ou refus* : pas de lieu de lavage, ou permission de regarder refusée (le refus est fusionné avec "aucun lieu", car dans les deux cas on ne peut pas affirmer qu'il y a un dispositif) ;
2. *Ustensile mobile* : seau, jarre, bouilloire ;
3. *Lieu fixe* : lavabo ou robinet, dans le logement ou dans la parcelle.

Les codes 6 ("autre raison") et 7 ("incohérent") deviennent des cases vides.

In [27]:
enfants["lavage_mains"] = recoder(brut["lavage_mains"], {
    4: "Aucun lieu ou refus", 5: "Aucun lieu ou refus",
    3: "Ustensile mobile",
    1: "Lieu fixe", 2: "Lieu fixe",
    6: None, 7: None})
compter(enfants["lavage_mains"])

lavage_mains
Aucun lieu ou refus    3453
Ustensile mobile       1057
Lieu fixe               390


**`telephone` - un membre du ménage a un téléphone portable.** Oui = 1, non = 0, "incohérent" = case vide.

In [28]:
enfants["telephone"] = recoder(brut["telephone"], {1: 1, 2: 0, 7: None})
compter(enfants["telephone"])

telephone
1.0    4239
0.0     661


## Étape 8 - Groupes, poids et colonnes d'audit

**Groupes.** On conserve :
- `grappe` : elle servira plus tard à découper les données par grappe entière. Les enfants d'une même grappe se ressemblent beaucoup ; si on les séparait au hasard, des enfants presque identiques se retrouveraient à la fois dans l'entraînement et dans le test, et les performances seraient gonflées ;
- `id_menage` : un identifiant unique de ménage, pour regrouper à la fin les enfants d'un même ménage.

**Poids d'enquête.** `poids_enfant` est le poids d'échantillonnage. Il sert **uniquement** pour les chiffres nationaux, jamais comme entrée de modèle.

**Colonnes d'audit.** Le milieu (urbain ou rural), le quintile de richesse et l'ethnie sont conservés mais ne seront **jamais des entrées**. Elles servent seulement à vérifier plus tard que l'outil n'est pas moins performant pour un groupe donné (équité). L'ethnie est une variable sensible.

In [29]:
enfants["id_menage"] = enfants["grappe"].astype(int).astype(str) + "-" + enfants["menage"].astype(int).astype(str)

assert (enfants["poids_enfant"] > 0).all(), "Un poids d'enquête est nul ou négatif"

enfants["milieu"] = recoder(brut["milieu"], {1: "Urbain", 2: "Rural"})

enfants["quintile_richesse"] = recoder(brut["quintile_richesse"], {
    1: "Le plus pauvre", 2: "Second", 3: "Moyen", 4: "Quatrième", 5: "Le plus riche"})

# Ethnie : codes de ch, libellés de hh (voir le piège signalé à l'étape 1). Le code 9 = ne sait pas.
libelles_ethnie = {1: "Adja-Ewe", 2: "Kabyè-Tem", 3: "Akposso-Akébou", 4: "Ana-Ifè",
                   5: "Para-Gourma-Akan", 6: "Autres Togolais", 7: "Autres nationalités", 9: None}
enfants["ethnie"] = recoder(brut["ethnie"], libelles_ethnie)

for v in ["milieu", "quintile_richesse", "ethnie"]:
    print(v); compter(enfants[v]); print()

milieu
milieu
Rural     3350
Urbain    1550

quintile_richesse
quintile_richesse
Le plus pauvre    1165
Second            1104
Moyen             1000
Quatrième          863
Le plus riche      768

ethnie
ethnie
Para-Gourma-Akan       1497
Adja-Ewe               1393
Kabyè-Tem              1307
Autres nationalités     263
Autres Togolais         223
Ana-Ifè                 117
Akposso-Akébou          100



In [30]:
enfants.dtypes

grappe               float64
menage               float64
ligne_enfant         float64
ligne_gardien        float64
age_mois             float64
haz                  float64
whz                  float64
haz_flag             float64
whz_flag             float64
edu_mere              object
acte_naissance       float64
milieu                object
quintile_richesse     object
ethnie                object
poids_enfant         float64
sexe_hl              float64
region                object
electricite           object
cuisiniere            object
source_eau           float64
toilettes             object
lavage_mains          object
telephone            float64
animaux              float64
taille_menage        float64
age_gardien          float64
stunt_niveau            int8
waste_niveau            int8
stunt                   int8
waste                   int8
sexe_f                 int64
eau_amelioree        float64
id_menage             object
dtype: object

## Étape 9 - Fixer les types

**Quoi.** Chaque colonne reçoit l'un des trois types suivants, et rien d'autre :

- **objet** (`object`) : les variables dont les valeurs sont du texte (région, cuisinière, instruction de la mère, électricité, toilettes, lavage des mains, milieu, quintile de richesse, ethnie) ;
- **entier** (`int64`) : les variables numériques **sans case vide** (âge, taille du ménage, oui/non 1/0, identifiants) ;
- **décimal** (`float64`) : les variables numériques **avec au moins une case vide**, car un entier classique ne peut pas contenir de case vide. Les z-scores et le poids d'enquête sont aussi des décimaux.

**Pourquoi.** On ne force pas de type catégoriel. À l'étape 7, les codes ont déjà été remplacés par des libellés, donc un modèle ne pourra plus lire la région 4 comme "deux fois" la région 2. C'est au moment du `ColumnTransformer` qu'on précisera, colonne par colonne, si un objet est une catégorie sans ordre (`OneHotEncoder`) ou ordonnée (`OrdinalEncoder` avec l'ordre explicite).

**Rappel des ordres (du plus faible au plus fort), pour plus tard :**
- `edu_mere` : Aucune, Primaire, Secondaire ou plus
- `electricite` : Non, Hors réseau, Réseau
- `toilettes` : Aucune, Non améliorée, Latrine améliorée, Chasse d'eau
- `lavage_mains` : Aucun lieu ou refus, Ustensile mobile, Lieu fixe
- `quintile_richesse` : Le plus pauvre, Second, Moyen, Quatrième, Le plus riche

In [31]:
# ---- Texte -> objet
colonnes_objet = ["region", "edu_mere", "electricite", "cuisiniere", "toilettes",
                  "lavage_mains", "milieu", "quintile_richesse", "ethnie"]
for col in colonnes_objet:
    enfants[col] = enfants[col].astype("object")

# ---- Numérique -> entier s'il n'y a aucune case vide, sinon décimal
colonnes_numeriques = ["age_mois", "age_gardien", "taille_menage",
                       "sexe_f", "acte_naissance", "eau_amelioree", "animaux", "telephone",
                       "grappe", "menage", "ligne_enfant"]
for col in colonnes_numeriques:
    enfants[col] = enfants[col].astype("int64" if enfants[col].notna().all() else "float64")

### Mise en forme finale du tableau

On range les colonnes dans un ordre logique : identifiants, cibles, 14 variables d'entrée, poids, colonnes d'audit. Les colonnes intermédiaires (codes bruts, drapeaux de plausibilité, numéros de ligne) disparaissent : leur rôle est terminé.

In [32]:
ENTREES_COMMUNES = ["age_mois", "region", "edu_mere", "electricite", "cuisiniere", "toilettes"]
ENTREES_RETARD = ["sexe_f", "acte_naissance", "eau_amelioree", "animaux"]
ENTREES_EMACIATION = ["age_gardien", "taille_menage", "lavage_mains", "telephone"]
ENTREES = ENTREES_COMMUNES + ENTREES_RETARD + ENTREES_EMACIATION

IDENTIFIANTS = ["grappe", "menage", "id_menage", "ligne_enfant"]
CIBLES = ["stunt", "waste"]
DESCRIPTION_CIBLES = ["stunt_niveau", "waste_niveau", "haz", "whz"]
POIDS = ["poids_enfant"]
AUDIT = ["milieu", "quintile_richesse", "ethnie"]

propre = enfants[IDENTIFIANTS + CIBLES + DESCRIPTION_CIBLES + ENTREES + POIDS + AUDIT].copy()
propre.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4900 entries, 0 to 4899
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   grappe             4900 non-null   int64  
 1   menage             4900 non-null   int64  
 2   id_menage          4900 non-null   object 
 3   ligne_enfant       4900 non-null   int64  
 4   stunt              4900 non-null   int8   
 5   waste              4900 non-null   int8   
 6   stunt_niveau       4900 non-null   int8   
 7   waste_niveau       4900 non-null   int8   
 8   haz                4900 non-null   float64
 9   whz                4900 non-null   float64
 10  age_mois           4900 non-null   int64  
 11  region             4900 non-null   object 
 12  edu_mere           4899 non-null   object 
 13  electricite        4900 non-null   object 
 14  cuisiniere         4898 non-null   object 
 15  toilettes          4887 non-null   object 
 16  sexe_f             4900 

## Étape 10 - Contrôles automatiques

**Quoi.** Une série de vérifications. **Si l'une d'elles échoue, le notebook s'arrête** avec un message clair.

**Pourquoi.** Si une règle est cassée un jour (par exemple en modifiant le notebook), on le sait immédiatement au lieu de découvrir des résultats étranges des semaines plus tard.

Les contrôles portent sur : le nombre de lignes, les doublons, les valeurs des variables 0/1, les types des colonnes, les cases vides (qui doivent être **exactement** celles attendues, calculées à partir des codes bruts), et la cohérence entre cibles et niveaux.

In [33]:
# 1) Nombre de lignes et doublons
assert len(propre) == nb_enfants, "Le nombre de lignes a changé"
assert not propre.duplicated(["grappe", "menage", "ligne_enfant"]).any(), "Doublon d'enfant"

# 2) Variables oui/non : uniquement 1 ou 0
for col in ["sexe_f", "acte_naissance", "eau_amelioree", "animaux", "telephone"]:
    assert propre[col].dropna().isin([0, 1]).all(), f"{col} contient autre chose que 0 et 1"

# 3) Les types sont ceux prévus : objet pour le texte, entier ou décimal pour le numérique
for col in colonnes_objet:
    assert propre[col].dtype == "object", f"{col} n'est pas de type objet"
for col in colonnes_numeriques:
    assert propre[col].dtype == ("int64" if propre[col].notna().all() else "float64"), f"{col} : mauvais type"

# 4) Les cases vides sont EXACTEMENT celles attendues.
#    Pour chaque variable : les codes bruts qu'on a décidé de vider, plus les cases déjà vides.
codes_vides = {
    "edu_mere": [9], "electricite": [4], "cuisiniere": [96], "toilettes": [96, 97],
    "acte_naissance": [7, 8, 9], "eau_amelioree": [96, 97], "animaux": [7],
    "age_gardien": [97], "taille_menage": [97], "lavage_mains": [6, 7],
    "telephone": [7], "ethnie": [9],
}
nom_brut = {"eau_amelioree": "source_eau"}  # les autres gardent le même nom
print("Cases vides par variable (attendu = trouvé) :")
for col, liste in codes_vides.items():
    b = brut[nom_brut.get(col, col)]
    attendu = int((b.isin(liste) | b.isna()).sum())
    trouve = int(propre[col].isna().sum())
    assert attendu == trouve, f"{col} : {trouve} cases vides trouvées, {attendu} attendues"
    print(f"   {col:<16} {trouve:>4}")

# Toutes les autres colonnes doivent être complètes
autres = [c for c in propre.columns if c not in codes_vides]
reste = propre[autres].isna().sum()
assert reste.sum() == 0, f"Cases vides inattendues : {reste[reste > 0].to_dict()}"
print("Toutes les autres colonnes sont complètes.")

# 5) Cohérence entre cibles, niveaux et z-scores
assert (propre["stunt"] == (propre["stunt_niveau"] >= 1)).all()
assert (propre["waste"] == (propre["waste_niveau"] >= 1)).all()
assert (propre["stunt"] == (propre["haz"] < -2)).all()
assert (propre["waste"] == (propre["whz"] < -2)).all()
assert set(propre["stunt"].unique()) <= {0, 1} and set(propre["waste"].unique()) <= {0, 1}

# 6) Bornes
assert propre["age_mois"].between(0, 59).all()
assert propre["taille_menage"].dropna().max() <= plafond_taille_menage
assert (propre["poids_enfant"] > 0).all()

print("\nTous les contrôles sont réussis.")

Cases vides par variable (attendu = trouvé) :
   edu_mere            1
   electricite         0
   cuisiniere          2
   toilettes          13
   acte_naissance     12
   eau_amelioree       9
   animaux             0
   age_gardien         0
   taille_menage       0
   lavage_mains        0
   telephone           0
   ethnie              0
Toutes les autres colonnes sont complètes.

Tous les contrôles sont réussis.



   edu_mere            1
   electricite         0
   cuisiniere          2
   toilettes          13
   acte_naissance     12
   eau_amelioree       9
   animaux             0
   age_gardien         0
   taille_menage       0
   lavage_mains        0
   telephone           0
   ethnie              0
Toutes les autres colonnes sont complètes.

Tous les contrôles sont réussis.


## Étape 11 - Exporter et relire

**Quoi.** On écrit un **seul fichier propre** au format parquet, puis on le relit et on le compare à ce qu'on voulait écrire.

**Pourquoi.** Le parquet conserve les types (texte, entiers, décimaux avec cases vides), contrairement au CSV. La relecture prouve que ce qui est écrit est identique à ce qu'on voulait écrire.

In [34]:
chemin_propre = DOSSIER_PROPRE / "nutripulse_clean.parquet"
propre.to_parquet(chemin_propre, index=False)

relu = pd.read_parquet(chemin_propre)
pd.testing.assert_frame_equal(propre, relu)   # s'arrête s'il y a la moindre différence
print("Relecture identique à l'original (valeurs et types).")

taille_ko = chemin_propre.stat().st_size / 1024
print(f"\nFichier écrit : {chemin_propre.relative_to(RACINE)}  ({taille_ko:.0f} Ko)")
print(f"Forme : {relu.shape[0]} enfants x {relu.shape[1]} colonnes")

Relecture identique à l'original (valeurs et types).

Fichier écrit : data\clean\nutripulse_clean.parquet  (93 Ko)
Forme : 4900 enfants x 28 colonnes


## Résumé du fichier propre

**Une ligne = un enfant de 0 à 59 mois** avec des mesures jugées fiables.

- **Identifiants** : `grappe`, `menage`, `id_menage`, `ligne_enfant`
- **Cibles** : `stunt` (retard de croissance), `waste` (émaciation), valeurs 0 ou 1
- **Pour décrire la réalité (jamais en entrée)** : `stunt_niveau`, `waste_niveau` (0 normal, 1 modéré, 2 sévère), `haz`, `whz`
- **Variables communes aux deux modèles (6)** : `age_mois`, `region`, `edu_mere`, `electricite`, `cuisiniere`, `toilettes`
- **Propres au retard de croissance (4)** : `sexe_f`, `acte_naissance`, `eau_amelioree`, `animaux`
- **Propres à l'émaciation (4)** : `age_gardien`, `taille_menage`, `lavage_mains`, `telephone`
- **Poids d'enquête** : `poids_enfant` (uniquement pour les chiffres nationaux)
- **Audit d'équité (jamais en entrée)** : `milieu`, `quintile_richesse`, `ethnie`

**Variables volontairement laissées de côté** : tout ce qui vient des mesures (poids, taille, IMC), l'ethnie et la religion comme entrées (sensibles), l'indice de richesse, le milieu et le type d'éclairage (redondants avec des variables déjà retenues), l'alimentation, la vaccination et les moustiquaires (trop de réponses absentes), et les symptômes récents comme la diarrhée ou la fièvre (risque de confondre cause et conséquence).

**Prochaine étape :** l'analyse exploratoire, qui ne lira que `data/clean/nutripulse_clean.parquet`.